In [ ]:
# =====================================================================
# BOLUM 0 - Ayarlar, klasor yapisi, grafik stili ve veri yukleme
# =====================================================================
import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")            # ekran acmadan dosyaya kaydetmek icin
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42

# ---- Portable project paths ---------------------------------------------
from pathlib import Path

def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "masonry_tower_primary_dataset.xlsx").is_file():
            return candidate
    raise FileNotFoundError(
        "Project root could not be located. Run this notebook from the repository "
        "root or from its code/ directory, and keep the data/ directory unchanged."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAT_FILE = DATA_DIR / "masonry_tower_primary_dataset.xlsx"
# ------------------------------------------------------------------------

# Cikti klasorleri
HIST_DIR    = os.path.join(OUTPUT_DIR, "Histograms")
BOX_DIR     = os.path.join(OUTPUT_DIR, "Boxplots")
CORR_DIR    = os.path.join(OUTPUT_DIR, "Correlation")
SCATTER_DIR = os.path.join(OUTPUT_DIR, "ScatterPlots")
for klasor in [OUTPUT_DIR, HIST_DIR, BOX_DIR, CORR_DIR, SCATTER_DIR]:
    os.makedirs(klasor, exist_ok=True)

# ---- Grafik ayarlari (makale kalitesi, sade tasarim) ------------------
DPI = 300
SAVE_PDF = False          # True yapilirsa her grafik ayrica PDF olarak da kaydedilir

sns.set_style("white")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "axes.linewidth": 0.9,
    "axes.grid": False,
    "figure.dpi": 110,
    "savefig.dpi": DPI,
    "savefig.bbox": "tight",
})

# Renk paleti (sade, baski dostu)
RENK_ANA = "#4C72B0"
RENK_KDE = "#C44E52"
RENK_CIZGI = "#333333"

def dosya_adi_temizle(ad):
    """Sutun adlarini Windows uyumlu dosya adina cevirir: 'Opening z/H' -> 'Opening_z_H'."""
    ad = ad.replace("/", "_").replace("%", "pct").replace("³", "3")
    ad = re.sub(r"[^\w\s-]", "", ad)         # parantez vb. isaretleri kaldir
    ad = re.sub(r"\s+", "_", ad.strip())
    return ad.strip("_")

def kaydet(fig, klasor, dosya_adi):
    """Figuru PNG (ve istenirse PDF) olarak kaydeder ve bellekten temizler."""
    yol_png = os.path.join(klasor, dosya_adi + ".png")
    fig.savefig(yol_png, dpi=DPI)
    if SAVE_PDF:
        fig.savefig(os.path.join(klasor, dosya_adi + ".pdf"))
    plt.close(fig)
    print("Kaydedildi:", yol_png)

# ---- Veri yukleme -----------------------------------------------------
df = pd.read_excel(MAT_FILE, sheet_name=0)

# Degisken tanimlari
GEO_COLS = ["Height (m)", "Section a (m)", "Section b (m)", "Wall Thickness (m)",
            "Opening z/H", "Opening Ratio x (%)", "Opening Ratio y (%)"]
MAT_COLS = ["E (MPa)", "d (kg/m3)"]
INPUT_COLS = GEO_COLS + MAT_COLS
TARGETS = ["f1 (Hz)", "f2 (Hz)"]
ALL_NUM = INPUT_COLS + TARGETS

# Grafiklerde kullanilacak okunakli Ingilizce eksen etiketleri
ETIKET = {
    "Height (m)": "Height, H (m)",
    "Section a (m)": "Section dimension a (m)",
    "Section b (m)": "Section dimension b (m)",
    "Wall Thickness (m)": "Wall thickness, t (m)",
    "Opening z/H": "Opening position, z/H (-)",
    "Opening Ratio x (%)": "Opening ratio in x-direction (%)",
    "Opening Ratio y (%)": "Opening ratio in y-direction (%)",
    "E (MPa)": "Modulus of elasticity, E (MPa)",
    "d (kg/m3)": "Density, $\\rho$ (kg/m$^3$)",
    "f1 (Hz)": "First natural frequency, $f_1$ (Hz)",
    "f2 (Hz)": "Second natural frequency, $f_2$ (Hz)",
}

# Geometry_ID (aykiri deger raporunda kayitlarin izlenebilmesi icin)
ROUND_DEC = 6
geo_key = df[GEO_COLS].round(ROUND_DEC).astype(str).agg("|".join, axis=1)
anahtar_to_id = {k: f"G{i+1:03d}" for i, k in enumerate(pd.unique(geo_key))}
df["Geometry_ID"] = geo_key.map(anahtar_to_id)

print(f"Veri yuklendi: {df.shape[0]} satir x {df.shape[1]} sutun")
print(f"Benzersiz geometri sayisi: {df['Geometry_ID'].nunique()}")

In [ ]:
# =====================================================================
# BOLUM 1 - Tanimlayici istatistikler (ana veri kumesi)
# =====================================================================

sayisal = df[ALL_NUM]
istatistik = pd.DataFrame({
    "Variable": sayisal.columns,
    "Count": sayisal.count().values,
    "Mean": sayisal.mean().values,
    "Std": sayisal.std().values,
    "Min": sayisal.min().values,
    "Q1": sayisal.quantile(0.25).values,
    "Median": sayisal.median().values,
    "Q3": sayisal.quantile(0.75).values,
    "Max": sayisal.max().values,
    "Skewness": sayisal.skew().values,
    "Kurtosis": sayisal.kurtosis().values,
})
istatistik.iloc[:, 1:] = istatistik.iloc[:, 1:].astype(float).round(4)

istatistik.to_excel(os.path.join(OUTPUT_DIR, "Descriptive_Statistics.xlsx"), index=False)
print("Descriptive_Statistics.xlsx olusturuldu.")

In [ ]:
# =====================================================================
# BOLUM 2 - Histogram + KDE grafikleri
# =====================================================================

BIN_SAYISI = 30

def panel_histogram(kolonlar, dosya_adi, ncols=3):
    """Birden fazla degisken icin tek figurde histogram + KDE paneli uretir."""
    n = len(kolonlar)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.4 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for i, kol in enumerate(kolonlar):
        ax = axes[i]
        sns.histplot(df[kol], bins=BIN_SAYISI, kde=True, color=RENK_ANA,
                     edgecolor="white", linewidth=0.5, ax=ax)
        if ax.lines:                       # KDE egrisini vurgula
            ax.lines[0].set_color(RENK_KDE)
            ax.lines[0].set_linewidth(1.8)
        ax.set_xlabel(ETIKET[kol])
        ax.set_ylabel("Frequency")
        ax.set_title(f"({chr(97 + i)})", loc="left", fontweight="bold")
        sns.despine(ax=ax)

    for j in range(n, len(axes)):          # kullanilmayan eksenleri gizle
        axes[j].set_visible(False)

    fig.tight_layout()
    kaydet(fig, HIST_DIR, dosya_adi)

# Geometrik ve malzeme girdileri ayri figur gruplari halinde
panel_histogram(GEO_COLS, "Fig_hist_kde_geometric_inputs", ncols=3)
panel_histogram(MAT_COLS, "Fig_hist_kde_material_inputs", ncols=2)

def tekil_histogram(kol, dosya_adi):
    """Tek bir degisken icin histogram + KDE grafigi."""
    fig, ax = plt.subplots(figsize=(6.0, 4.2))
    sns.histplot(df[kol], bins=BIN_SAYISI, kde=True, color=RENK_ANA,
                 edgecolor="white", linewidth=0.5, ax=ax)
    if ax.lines:
        ax.lines[0].set_color(RENK_KDE)
        ax.lines[0].set_linewidth(2.0)
    ax.set_xlabel(ETIKET[kol])
    ax.set_ylabel("Frequency")
    sns.despine(ax=ax)
    fig.tight_layout()
    kaydet(fig, HIST_DIR, dosya_adi)

# Hedef degiskenler icin ayri histogramlar
tekil_histogram("f1 (Hz)", "Fig_hist_f1")
tekil_histogram("f2 (Hz)", "Fig_hist_f2")

In [ ]:
# =====================================================================
# BOLUM 3 - f1 ve f2 icin kutu grafikleri
# =====================================================================

def kutu_grafigi(kol, dosya_adi):
    """Tek degisken icin dusey kutu grafigi (aykiri degerler isaretli)."""
    fig, ax = plt.subplots(figsize=(3.6, 4.6))
    sns.boxplot(y=df[kol], color=RENK_ANA, width=0.35, ax=ax,
                fliersize=3.5, linewidth=1.1,
                flierprops=dict(marker="o", markerfacecolor="none",
                                markeredgecolor=RENK_KDE, markeredgewidth=0.8))
    ax.set_ylabel(ETIKET[kol])
    ax.set_xlabel("")
    ax.set_xticks([])
    sns.despine(ax=ax, bottom=True)
    fig.tight_layout()
    kaydet(fig, BOX_DIR, dosya_adi)

kutu_grafigi("f1 (Hz)", "Fig_boxplot_f1")
kutu_grafigi("f2 (Hz)", "Fig_boxplot_f2")

In [ ]:
# =====================================================================
# BOLUM 4 - Pearson ve Spearman korelasyon analizi
# =====================================================================

pearson  = df[ALL_NUM].corr(method="pearson").round(4)
spearman = df[ALL_NUM].corr(method="spearman").round(4)

# Girdi degiskenlerinin hedeflerle korelasyonlari (ayri tablo)
hedef_pearson = pearson.loc[INPUT_COLS, TARGETS].copy()
hedef_pearson.columns = ["Pearson_with_f1", "Pearson_with_f2"]
hedef_spearman = spearman.loc[INPUT_COLS, TARGETS].copy()
hedef_spearman.columns = ["Spearman_with_f1", "Spearman_with_f2"]

# Yuksek korelasyonlu girdi ciftleri (esik degeri degistirilebilir)
KORELASYON_ESIGI = 0.80
ciftler = []
for i in range(len(INPUT_COLS)):
    for j in range(i + 1, len(INPUT_COLS)):
        r = pearson.loc[INPUT_COLS[i], INPUT_COLS[j]]
        if abs(r) >= KORELASYON_ESIGI:
            ciftler.append({"Variable_1": INPUT_COLS[i], "Variable_2": INPUT_COLS[j],
                            "Pearson_r": r,
                            "Spearman_rho": spearman.loc[INPUT_COLS[i], INPUT_COLS[j]]})
highly_correlated = pd.DataFrame(ciftler)

# Pearson ile Spearman arasindaki mutlak fark (iki yontemin yakinligini gormek icin)
fark = (pearson - spearman).abs().round(4)

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Pearson_Correlation.xlsx"),
                    engine="openpyxl") as writer:
    pearson.to_excel(writer, sheet_name="Pearson_matrix")
    hedef_pearson.to_excel(writer, sheet_name="Correlation_with_targets")
    highly_correlated.to_excel(writer, sheet_name="Highly_correlated_pairs", index=False)

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Spearman_Correlation.xlsx"),
                    engine="openpyxl") as writer:
    spearman.to_excel(writer, sheet_name="Spearman_matrix")
    hedef_spearman.to_excel(writer, sheet_name="Correlation_with_targets")
    fark.to_excel(writer, sheet_name="Abs_diff_Pearson_Spearman")

print("Korelasyon Excel dosyalari olusturuldu.")
print("Pearson-Spearman maksimum mutlak fark:",
      fark.where(~np.eye(len(fark), dtype=bool)).max().max())

# ---- Pearson isi haritasi --------------------------------------------
kisa_ad = {c: ETIKET[c].split(",")[0].split(" (")[0] for c in ALL_NUM}
pearson_gorsel = pearson.rename(index=kisa_ad, columns=kisa_ad)

fig, ax = plt.subplots(figsize=(9.5, 8.0))
sns.heatmap(pearson_gorsel, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
            linecolor="white", annot_kws={"size": 8.5},
            cbar_kws={"label": "Pearson correlation coefficient", "shrink": 0.8}, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
fig.tight_layout()
kaydet(fig, CORR_DIR, "Fig_pearson_heatmap")

# ---- Hedeflerle korelasyonlarin sirali cubuk grafikleri ---------------
def korelasyon_bar(hedef, dosya_adi):
    """Girdi degiskenlerinin belirtilen hedefle Pearson korelasyonunu sirali cizer."""
    seri = pearson.loc[INPUT_COLS, hedef].sort_values()
    renkler = [RENK_KDE if v < 0 else RENK_ANA for v in seri.values]

    fig, ax = plt.subplots(figsize=(7.2, 5.0))
    ax.barh([ETIKET[k] for k in seri.index], seri.values,
            color=renkler, edgecolor="white", height=0.65)
    ax.axvline(0, color=RENK_CIZGI, linewidth=0.9)
    ax.set_xlabel(f"Pearson correlation coefficient with {ETIKET[hedef]}")
    ax.set_xlim(-1, 1)

    # Deger etiketleri
    for y, v in enumerate(seri.values):
        ax.text(v + (0.03 if v >= 0 else -0.03), y, f"{v:.2f}",
                va="center", ha="left" if v >= 0 else "right", fontsize=9)
    sns.despine(ax=ax)
    fig.tight_layout()
    kaydet(fig, CORR_DIR, dosya_adi)

korelasyon_bar("f1 (Hz)", "Fig_correlation_bar_f1")
korelasyon_bar("f2 (Hz)", "Fig_correlation_bar_f2")

In [ ]:
# =====================================================================
# BOLUM 5 - Secilmis sacilim grafikleri (pairplot uretilmez)
# =====================================================================

TREND_CIZGISI = True      # doğrusal eğilim çizgisi eklensin mi
ANNOTATE_R = True         # grafik uzerine Pearson r degeri yazilsin mi

SCATTER_CIFTLERI = [
    ("Height (m)",   "f1 (Hz)", "Fig_scatter_Height_f1"),
    ("Height (m)",   "f2 (Hz)", "Fig_scatter_Height_f2"),
    ("E (MPa)",      "f1 (Hz)", "Fig_scatter_E_f1"),
    ("E (MPa)",      "f2 (Hz)", "Fig_scatter_E_f2"),
    ("d (kg/m3)",    "f1 (Hz)", "Fig_scatter_Density_f1"),
    ("d (kg/m3)",    "f2 (Hz)", "Fig_scatter_Density_f2"),
    ("Opening z/H",  "f1 (Hz)", "Fig_scatter_OpeningZH_f1"),
    ("Opening z/H",  "f2 (Hz)", "Fig_scatter_OpeningZH_f2"),
    ("f1 (Hz)",      "f2 (Hz)", "Fig_scatter_f1_f2"),
]

def sacilim_grafigi(x_kol, y_kol, dosya_adi):
    """Iki degisken arasinda sacilim grafigi; istege bagli dogrusal egilim cizgisi."""
    x = df[x_kol].values
    y = df[y_kol].values

    fig, ax = plt.subplots(figsize=(5.6, 4.6))
    ax.scatter(x, y, s=16, color=RENK_ANA, alpha=0.55,
               edgecolors="none", zorder=2)

    if TREND_CIZGISI:
        katsayi = np.polyfit(x, y, 1)                  # yalnizca dogrusal egilim
        x_cizgi = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_cizgi, np.polyval(katsayi, x_cizgi),
                color=RENK_KDE, linewidth=1.8, zorder=3, label="Linear trend")
        ax.legend(frameon=False, loc="best")

    if ANNOTATE_R:
        r = np.corrcoef(x, y)[0, 1]
        ax.text(0.03, 0.95, f"r = {r:.2f}", transform=ax.transAxes,
                va="top", ha="left", fontsize=10,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                          edgecolor="0.8", alpha=0.85))

    ax.set_xlabel(ETIKET[x_kol])
    ax.set_ylabel(ETIKET[y_kol])
    sns.despine(ax=ax)
    fig.tight_layout()
    kaydet(fig, SCATTER_DIR, dosya_adi)

for x_kol, y_kol, ad in SCATTER_CIFTLERI:
    sacilim_grafigi(x_kol, y_kol, ad)

In [ ]:
# =====================================================================
# BOLUM 6 - IQR yontemi ile aykiri deger analizi
# ONEMLI: Hicbir satir silinmez, hicbir donusum uygulanmaz. Sadece rapor.
# =====================================================================

IQR_KATSAYISI = 1.5

ozet_satirlari = []
bayrak = pd.DataFrame(False, index=df.index, columns=ALL_NUM)

for kol in ALL_NUM:
    q1 = df[kol].quantile(0.25)
    q3 = df[kol].quantile(0.75)
    iqr = q3 - q1
    alt = q1 - IQR_KATSAYISI * iqr
    ust = q3 + IQR_KATSAYISI * iqr

    alt_mask = df[kol] < alt
    ust_mask = df[kol] > ust
    bayrak[kol] = alt_mask | ust_mask

    ozet_satirlari.append({
        "Variable": kol,
        "Q1": round(q1, 4), "Q3": round(q3, 4), "IQR": round(iqr, 4),
        "Lower_bound": round(alt, 4), "Upper_bound": round(ust, 4),
        "Outliers_below": int(alt_mask.sum()),
        "Outliers_above": int(ust_mask.sum()),
        "Total_outliers": int((alt_mask | ust_mask).sum()),
        "Percentage_%": round(100 * (alt_mask | ust_mask).sum() / len(df), 2),
    })

outlier_summary = pd.DataFrame(ozet_satirlari)

# Herhangi bir degiskende aykiri isaretlenen kayitlar
bayrak_sayisi = bayrak.sum(axis=1)
etkilenen_degiskenler = bayrak.apply(
    lambda satir: ", ".join(bayrak.columns[satir.values]), axis=1)

flagged = df.loc[bayrak_sayisi > 0].copy()
flagged["Number_of_flagged_variables"] = bayrak_sayisi[bayrak_sayisi > 0]
flagged["Flagged_variables"] = etkilenen_degiskenler[bayrak_sayisi > 0]

# Aykiri kayitlarin geometri bazinda dagilimi
geometri_dagilimi = (flagged.groupby("Geometry_ID").size()
                     .reset_index(name="Flagged_record_count")
                     .sort_values("Flagged_record_count", ascending=False))

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Outlier_Report.xlsx"),
                    engine="openpyxl") as writer:
    outlier_summary.to_excel(writer, sheet_name="Outlier_Summary", index=False)
    flagged.to_excel(writer, sheet_name="Flagged_Records", index=False)
    bayrak.astype(int).to_excel(writer, sheet_name="Outlier_Flag_Matrix", index=True)
    geometri_dagilimi.to_excel(writer, sheet_name="Flagged_by_Geometry", index=False)

print("\nOutlier_Report.xlsx olusturuldu.")
print(f"Herhangi bir degiskende aykiri isaretlenen kayit sayisi: {len(flagged)} / {len(df)}")
print("\nAykiri deger ozeti:")
print(outlier_summary.to_string(index=False))
print("\nNOT: Hicbir satir silinmemis, hicbir donusum uygulanmamistir.")
print("\nAsama 2 tamamlandi. Cikti klasoru:", OUTPUT_DIR)